In [25]:
import os
import numpy as np
import pandas as pd

EMBEDDING_DIR = "/home/pc/LSC24_SemanticSearchWebApp/backend/data/embeddings/keyframe_clips/MVK2"
COLLECTION_NAME = "vbs25_mvk_clips"
ID_MAPPING = "/home/pc/LSC24_SemanticSearchWebApp/backend/data/id_mapping/mvk/keyframes.csv"

In [26]:
from pymilvus import connections, utility, MilvusException, Collection
connections.connect(host="localhost", port="19530")
try:
    collections = utility.list_collections()
    print("List of collections: ", collections)
except MilvusException as e:
    print(e)


from pymilvus import MilvusClient, DataType
CLUSTER_ENDPOINT = "http://localhost:19530"
TOKEN = "root:Milvus"
client = MilvusClient(uri=CLUSTER_ENDPOINT, token=TOKEN)

List of collections:  ['vbs25_v3c_clips', 'vbs25_mvk_clips', 'vbs25_lhe_clips', 'vbs25_bgem3', 'vbs25_clips']


In [29]:
collection = Collection('vbs25_mvk_clips')
collection.num_entities
client.get_collection_stats(collection_name="vbs25_mvk_clips")

{'row_count': 112682}

In [10]:
# if not os.path.exists(ID_MAPPING):
#     df = pd.DataFrame(columns=["keyframe_name", "keyframe_id"])
#     df.to_csv(ID_MAPPING, index=False)
#     print(f"Created {ID_MAPPING}")

In [11]:
# id_mapping = pd.read_csv(ID_MAPPING)
# id_mapping.set_index("keyframe_name", inplace=True)
# CURR_ID = len(id_mapping)
# print("Current ID: ", CURR_ID)

In [28]:
import math

def get_keyframe_order_from_name(name):
    return int(name.split('.')[0])

def get_keyframe_name_from_path(path):
    keyframe_id = int(path.split("/")[-1].split(".")[0])
    context_id = math.ceil(keyframe_id / 16)
    video_id = path.split("/")[-2]
    return f"MVK/{video_id}/{context_id:05d}/{keyframe_id:05d}"

id_mapping = pd.read_csv(ID_MAPPING, index_col=0)

id_mapping_updates = []

for entry in sorted(os.scandir(EMBEDDING_DIR), key=lambda e: e.name):
    print(entry.name)
    if entry.is_dir():
        clip_name = entry.name
        clip_dir = os.path.join(EMBEDDING_DIR, clip_name)
        for clip_entry in sorted(os.scandir(clip_dir), key=lambda e: get_keyframe_order_from_name(e.name)):
            if clip_entry.is_file() and clip_entry.name.endswith('.npy'):
                try:
                    name = clip_entry.name
                    path = os.path.join(clip_dir, name)
                    keyframe_name = get_keyframe_name_from_path(path)
                    keyframe_id = id_mapping.loc[keyframe_name, "keyframe_id"]
                    embedding = np.load(path).astype(np.float32)

                    # Insert data to Milvus
                    data = [{
                        "keyframe_id": id_mapping.loc[keyframe_name, "keyframe_id"],
                        "keyframe_name": keyframe_name,
                        "embedding": embedding,
                    }]
                    # collection.insert(data=data)
                    res = client.insert(collection_name=COLLECTION_NAME, data=data)
                    # print(res)
                    # print(data[0]["keyframe_name"], data[0]["keyframe_id"])

                except:
                    with open("mvk_error.log", "a") as f:
                        f.write(f"{path}\n")
                    f.close()
            else:
                print("Not a file")

        print(f"Inserted {clip_name}")

BigIsland1_Jan2023_0001
Inserted BigIsland1_Jan2023_0001
BigIsland1_Jan2023_0002
Inserted BigIsland1_Jan2023_0002
BigIsland1_Jan2023_0003
Inserted BigIsland1_Jan2023_0003
BigIsland1_Jan2023_0004
Inserted BigIsland1_Jan2023_0004
BigIsland1_Jan2023_0005
Inserted BigIsland1_Jan2023_0005
BigIsland1_Jan2023_0006
Inserted BigIsland1_Jan2023_0006
BigIsland1_Jan2023_0007
Inserted BigIsland1_Jan2023_0007
BigIsland1_Jan2023_0008
Inserted BigIsland1_Jan2023_0008
BigIsland1_Jan2023_0009
Inserted BigIsland1_Jan2023_0009
BigIsland1_Jan2023_0010
Inserted BigIsland1_Jan2023_0010
BigIsland1_Jan2023_0011
Inserted BigIsland1_Jan2023_0011
BigIsland1_Jan2023_0012
Inserted BigIsland1_Jan2023_0012
BigIsland1_Jan2023_0013
Inserted BigIsland1_Jan2023_0013
BigIsland1_Jan2023_0014
Inserted BigIsland1_Jan2023_0014
BigIsland1_Jan2023_0015
Inserted BigIsland1_Jan2023_0015
BigIsland1_Jan2023_0016
Inserted BigIsland1_Jan2023_0016
BigIsland1_Jan2023_0017
Inserted BigIsland1_Jan2023_0017
BigIsland1_Jan2023_0018
Inserte

In [24]:
# client.get(collection_name="vbs25_clips", ids=[1, 2, 3])[0]['embedding'].__len__()
client.get(collection_name="vbs25_mvk_clips", ids=[3957])

data: ["{'keyframe_id': 3957, 'keyframe_name': 'MVK/Ambon_Apr2012_0001/00001/00001', 'embedding': [0.008584865, 0.059366528, -0.028184552, -0.01104392, -0.01596203, -0.037657004, 0.019803394, -0.082763925, -0.018100971, -0.033117212, 0.11378585, 0.044204783, -0.0030392609, -0.010352766, -0.028926633, -0.029013935, 0.010614677, 0.009894421, -0.04062533, -0.02629297, -0.005038152, -0.008592141, -0.00052609586, 0.011327658, 0.0024917936, 0.08334595, -0.011393135, -0.00021303017, 0.007457193, -0.31522462, 0.003152028, -0.027384266, 0.011604119, -0.017577149, 0.031080125, -0.08229831, 8.93499e-05, 0.013612105, -0.02368841, -0.011829654, -0.0128481975, -0.062451262, -0.011967885, -0.023644758, 0.027733482, 0.014885284, -0.0065259533, -0.030177986, -0.019206818, 0.056194495, 0.0042415056, 0.05724214, 0.023615656, -0.0072389334, 0.028737474, 0.033873845, 0.05139279, 0.0004938117, -0.018857604, 0.0155109605, 0.040887244, -0.0012140673, 0.0036012784, -0.013095558, -0.019061312, 0.012571735, 0.05